<h1 style='color:#A52A2A'>Generating Paligrams</h1> 
<h3 style='color:#A52A2A'>A companion to <small><a href="https://oeis.org/A400282">A400282</a></small>.</h3>
<h4 style='color:#708090'>Peter Luschny, October 2026</h4>

A palindromic diagram of n is an edge-connected, self-conjugate skew Young/Ferrers diagram with n cells, if the string formed by concatenating the rows is a palindromic word over a two-symbol alphabet. We suggest calling these objects 'paligrams', and we will use this new name here.

For example, the following palindromic diagram with 10 cells

```python
  o o ● ● 
  o ● ● ● 
  ● ● ● o 
  ● ● o o 
```
forms the palindromic string ```oo●●o●●●●o●●oo``` when read by rows. Reversing the concatenated string reverses the row order and each row.

Self-conjugacy means the diagram is invariant under reflection in the main diagonal (transposition), and palindromicity means it is invariant under the half-turn (rotation by 180 degrees). Together, these force a third symmetry, the reflection in the antidiagonal. Hence a paligram is invariant under the full Klein four-group (Vierergruppe) generated by transposition and the half-turn, acting on its bounding box. The bounding box, i.e., the smallest rectangle containing the diagram, is a square. Thus a paligram is symmetric with respect to both diagonals of this square and under the half-turn about its centre.

Each row is a single contiguous nonempty run of ```●``` cells (this follows from the skew shape and the edge-connectivity), and by symmetry the same holds for each column. The top row touches the right edge, and the bottom row touches the left edge. As one moves downward, both ends of the run move weakly left, and consecutive runs share a column.

Theorem: For each n >= 19 at least one paligram exists. (No paligrams exist for n in {0, 2, 3, 5, 6, 8, 11, 12, 18}.)

## The core functions for generating paligrams

In [118]:
from math import isqrt
from typing import Generator, LiteralString

def hooks(h, m, i=1, cap=10**9):
    """Strictly decreasing a_1 > a_2 > ... >= 0, a_i <= h-1-2i, sum(2*a_i+1) = m."""
    if m == 0: yield ()
    for a in range(min(cap, h - 1 - 2 * i, (m - 1) // 2), isqrt(max(m - 1, 0)) - 1, -1):
        for t in hooks(h, m - 2 * a - 1, i + 1, a - 1):
            yield (a,) + t

def paligrams(n: int, cell="●", exterior="○") -> Generator[tuple[list[int], list[str], int], int, None]:
    """Yield (widths, rows) of all palindromic diagrams with n cells."""
    for h in range(1, (n + 2) // 3 + 1):  # n >= 3h-2
        m = h * h - n                     # n = h^2 - 2|mu|
        if m < 0 or m % 2: continue
        found = []; A = [()]
        for A in hooks(h, m // 2):        # mu = self-conjugate, given by its hooks
            S = {(i, j) for i, a in enumerate(A, 1) for j in range(1, a + i + 1)}
            S |= {(j, i) for i, j in S}   # add the columns (transpose)
            mu = [sum((r, c) in S for c in range(1, h + 1)) for r in range(1, h + 1)]
            found.append((tuple((mu[r] + 1, h - mu[h - 1 - r]) for r in range(h)), A))
        for rows, A in sorted(found):
            yield ([R - L + 1 for L, R in rows],
                   [''.join(cell if L <= c <= R else exterior for c in range(1, h + 1)) for L, R in rows],
                   A)

## The connection with Dyck paths

In [119]:
def first_half(h, A) -> LiteralString:
    """(h, A) -> First half: h steps, D at position h - a for every a in A."""
    return ''.join('D' if h - j in A else 'U' for j in range(1, h + 1))

def full_path(half) -> LiteralString:
    """First half -> the second half is the first reversed with U and D swapped."""
    return half + half[::-1].translate(str.maketrans('UD', 'DU'))

def frobenius(half) -> tuple[int, tuple[int, ...]]:
    """Inverse of first_half: (h, A)."""
    h = len(half)
    return h, tuple(h - j for j, s in enumerate(half, 1) if s == 'D')

def build(h, A, cell="●", exterior="○"):
    """(h, A) -> (widths, rows) of the paligram with side h and Frobenius sequence A."""
    S = {(i, j) for i, a in enumerate(A, 1) for j in range(1, a + i + 1)}
    S |= {(j, i) for i, j in S}             # mu: self-conjugate, arms = legs = A
    mu = [sum((r, c) in S for c in range(1, h + 1)) for r in range(1, h + 1)]
    LR = [(mu[r] + 1, h - mu[h - 1 - r]) for r in range(h)] # row r runs from L to R
    return ([R - L + 1 for L, R in LR],
            [''.join(cell if L <= c <= R else exterior for c in range(1, h + 1)) for L, R in LR])
    
def from_path(path):
    """Dyck path -> (h, A). Raises ValueError if not a symmetric primitive Dyck path."""
    h = len(path) // 2
    height = [0]
    for s in path: height.append(height[-1] + (1 if s == 'U' else -1))
    if len(path) != 2 * h or set(path) - set('UD') or height[-1] != 0 or min(height[1:-1], default=1) < 1:
        raise ValueError("not a primitive Dyck path")
    if path != full_path(path[:h]):
        raise ValueError("not symmetric (second half is not the mirror of the first)")
    return frobenius(path[:h])

#### A display function with several format options

In [138]:
def show_paligrams(
    n: int, 
    flat: bool=False, 
    info: bool=False, 
    dyck: bool=False, 
    string: bool=False):

    p = list(paligrams(n))
    if not p: return

    if string:
        for l in p: 
            s = ''.join(l[1])
            print(n, s)
        return
    
    if dyck:
        for l in p:
            half = first_half(len(l[0]), l[2])
            print(n, half)
        return

    if flat: 
        for l in p:
            half = first_half(len(l[0]), l[2]) 
            print(f"{n:4} {l[1]}", half)
        return

    for l in p:
        if not l: continue
        if info:
            half = first_half(len(l[0]), l[2])
            print(n, len(l[0]), l[2], l[0], half)
            # round trip
            # print(full_path(half))
            # print(frobenius(half))
        else:
            print(n, len(l[0]))
        for s in l[1]:
            print("  ", s)

#### Examples of using the display function

In [132]:
show_paligrams(27, string=True)

27 ○○○○●●●○○○●●●●○○○●●●●○●●●●●○●●●●○○○●●●●○○○●●●○○○○
27 ○○○○○○○●●○○○○○○●●●○○○○○●●●○○○○●●●●○○○○○●●●○○○○○●●●●○○○○●●●○○○○○●●●○○○○○○●●○○○○○○○


In [125]:
# Example: show paligrams as squares (this is the std. format).
show_paligrams(27)

27 7
   ○○○○●●●
   ○○○●●●●
   ○○○●●●●
   ○●●●●●○
   ●●●●○○○
   ●●●●○○○
   ●●●○○○○
27 9
   ○○○○○○○●●
   ○○○○○○●●●
   ○○○○○●●●○
   ○○○●●●●○○
   ○○○●●●○○○
   ○○●●●●○○○
   ○●●●○○○○○
   ●●●○○○○○○
   ●●○○○○○○○


In [145]:
# Example: show paligrams as squares plus information about structure (expert mode).
# | n | h = the height/width of the square | Frobenius coordinates | row occupancy | half Dyck path |

show_paligrams(32, info=True)

32 8 (4, 3) [3, 3, 6, 4, 4, 6, 3, 3] UUUDDUUU
   ○○○○○●●●
   ○○○○○●●●
   ○○●●●●●●
   ○○●●●●○○
   ○○●●●●○○
   ●●●●●●○○
   ●●●○○○○○
   ●●●○○○○○
32 8 (5, 2) [2, 4, 5, 5, 5, 5, 4, 2] UUDUUDUU
   ○○○○○○●●
   ○○○○●●●●
   ○○●●●●●○
   ○○●●●●●○
   ○●●●●●○○
   ○●●●●●○○
   ●●●●○○○○
   ●●○○○○○○
32 10 (6, 5, 3, 1) [3, 3, 4, 3, 3, 3, 3, 4, 3, 3] UUUDDUDUDU
   ○○○○○○○●●●
   ○○○○○○○●●●
   ○○○○○○●●●●
   ○○○○○●●●○○
   ○○○○●●●○○○
   ○○○●●●○○○○
   ○○●●●○○○○○
   ●●●●○○○○○○
   ●●●○○○○○○○
   ●●●○○○○○○○
32 10 (7, 4, 3, 1) [2, 4, 3, 4, 3, 3, 4, 3, 4, 2] UUDUUDDUDU
   ○○○○○○○○●●
   ○○○○○○●●●●
   ○○○○○○●●●○
   ○○○○○●●●●○
   ○○○○●●●○○○
   ○○○●●●○○○○
   ○●●●●○○○○○
   ○●●●○○○○○○
   ●●●●○○○○○○
   ●●○○○○○○○○
32 10 (7, 5, 2, 1) [2, 3, 4, 3, 4, 4, 3, 4, 3, 2] UUDUDUUDDU
   ○○○○○○○○●●
   ○○○○○○○●●●
   ○○○○○●●●●○
   ○○○○○●●●○○
   ○○○○●●●●○○
   ○○●●●●○○○○
   ○○●●●○○○○○
   ○●●●●○○○○○
   ●●●○○○○○○○
   ●●○○○○○○○○
32 10 (7, 5, 3, 0) [2, 3, 3, 4, 4, 4, 4, 3, 3, 2] UUDUDUDUUD
   ○○○○○○○○●●
   ○○○○○○○●●●
   ○○○○○○●●●○
   ○○○○●●●

In [144]:
# Example: half Dyck paths

show_paligrams(37, dyck=True)

37 UUUUDUD
37 UUUUDDDUD
37 UUUDUDUDD
37 UUDUUUDDD
37 UUDUDUUUU
37 UUUDDUDUDUU
37 UUDUUDDUDUU
37 UUDUDUUDDUU
37 UUDUDUDUUDU
37 UUDUDUDUDUDUD


In [ ]:
# Examples: show paligrams as lists of rows, plus half Dyck path.

for n in range(26):
    show_paligrams(n, flat=True)

   1 ['●'] U
   4 ['●●', '●●'] UU
   7 ['○●●', '●●●', '●●○'] UUD
   9 ['●●●', '●●●', '●●●'] UUU
  10 ['○○●●', '○●●●', '●●●○', '●●○○'] UUDU
  13 ['○○○●●', '○○●●●', '○●●●○', '●●●○○', '●●○○○'] UUDUD
  14 ['○●●●', '●●●●', '●●●●', '●●●○'] UUUD
  15 ['○○○●●', '○●●●●', '○●●●○', '●●●●○', '●●○○○'] UUDUU
  16 ['●●●●', '●●●●', '●●●●', '●●●●'] UUUU
  16 ['○○○○●●', '○○○●●●', '○○●●●○', '○●●●○○', '●●●○○○', '●●○○○○'] UUDUDU
  17 ['○○●●●', '○○●●●', '●●●●●', '●●●○○', '●●●○○'] UUUDD
  19 ['○○●●●', '○●●●●', '●●●●●', '●●●●○', '●●●○○'] UUUDU
  19 ['○○○○○●●', '○○○○●●●', '○○○●●●○', '○○●●●○○', '○●●●○○○', '●●●○○○○', '●●○○○○○'] UUDUDUD
  20 ['○○○●●●', '○○○●●●', '○○●●●●', '●●●●○○', '●●●○○○', '●●●○○○'] UUUDDU
  20 ['○○○○●●', '○○●●●●', '○●●●●○', '○●●●●○', '●●●●○○', '●●○○○○'] UUDUUD
  21 ['○○○○○●●', '○○○○●●●', '○○●●●●○', '○○●●●○○', '○●●●●○○', '●●●○○○○', '●●○○○○○'] UUDUDUU
  22 ['○○○○●●', '○●●●●●', '○●●●●○', '○●●●●○', '●●●●●○', '●●○○○○'] UUDUUU
  22 ['○○○○○○●●', '○○○○○●●●', '○○○○●●●○', '○○○●●●○○', '○○●●●○○○', '○●●●○○